# 🏥 Hospital Readmission Prediction — Data Preprocessing Walkthrough
### HealthBridge Medical Center | Advanced Data Analytics Capstone

---

## 📋 What This Notebook Does
This notebook walks through **every preprocessing step** needed to prepare the UCI Diabetes 130-US Hospitals dataset for machine learning. Each step includes:
- ✅ A plain-English explanation of **what** we're doing
- ✅ A plain-English explanation of **why** we're doing it
- ✅ The actual Python code
- ✅ What to look for in the output

## 📁 Dataset Download
Before running this notebook, download the dataset from:
> https://archive.ics.uci.edu/dataset/296/diabetes+130-us-hospitals-for-years-1999-2008

You need two files:
- `diabetic_data.csv` — the main patient encounter data
- `IDs_mapping.csv` — lookup table for coded values

Place both files in the **same folder as this notebook**.

---

## STEP 0 — Import Libraries

**What:** Load all the Python tools we'll need for this project.

**Why:** Python itself is like a blank toolbox. Libraries like `pandas` add tools for working with tables of data, `matplotlib` adds charting tools, and `sklearn` adds machine learning tools. We load them all upfront so they're ready to use.

In [ ]:
# ── Core data tools ──────────────────────────────────────────────────────────
import pandas as pd          # Think of this as Python's Excel — handles tables (DataFrames)
import numpy as np           # Math/numbers toolkit — used behind the scenes constantly

# ── Visualization tools ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt   # Basic charting library
import seaborn as sns             # prettier, easier charts built on top of matplotlib

# ── Machine learning tools (we'll use these later in the project) ─────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)   # Show up to 50 columns at once
pd.set_option('display.max_rows', 100)     # Show up to 100 rows at once
pd.set_option('display.float_format', '{:.2f}'.format)  # Round decimals to 2 places

# Make charts look nice
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ All libraries loaded successfully!')

---
## STEP 1 — Load the Data

**What:** Read the CSV file into Python as a DataFrame (a table).

**Why:** Until we load the data, Python has no idea what's in the file. `pd.read_csv()` reads it and creates a DataFrame — think of it like importing a spreadsheet into Python where every row is a patient encounter and every column is a variable.

**What to look for in the output:**
- How many rows (patient encounters) and columns (variables) are there?
- Do the column names look right?

In [ ]:
# ── Load the main dataset ─────────────────────────────────────────────────────
# UPDATE THIS PATH if your file is in a different location
df = pd.read_csv('diabetic_data.csv')

# ── Quick first look ──────────────────────────────────────────────────────────
print(f'📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   → Each row = one patient hospital encounter')
print(f'   → Each column = one data variable')
print()
print('📋 Column names:')
print(df.columns.tolist())

In [ ]:
# ── Look at the first 5 rows ──────────────────────────────────────────────────
# .head() shows the first N rows — great for getting a feel for the data
print('First 5 rows of the dataset:')
df.head()

In [ ]:
# ── Check data types and non-null counts ──────────────────────────────────────
# .info() is like a summary report: tells you the type of each column
# and how many values are NOT missing
# 'object' dtype = text/categorical  |  'int64'/'float64' = numbers
df.info()

In [ ]:
# ── Statistical summary of numeric columns ────────────────────────────────────
# .describe() gives you count, mean, min, max, percentiles for numeric columns
# Look for anything that seems wrong: negative ages? Huge outliers?
df.describe()

---
## STEP 2 — Understand Our Target Variable: `readmitted`

**What:** Look at the column we're trying to predict — `readmitted`.

**Why:** Before any preprocessing, we need to understand what we're predicting. The `readmitted` column has 3 values:
- `<30` → Patient was readmitted within 30 days (**this is what we want to predict — our 'positive' class**)
- `>30` → Patient was readmitted after 30 days (NOT within our HRRP window)
- `NO`  → Patient was NOT readmitted

We'll convert this into a **binary** (0 or 1) variable:
- `1` = Readmitted within 30 days (HIGH RISK)
- `0` = Not readmitted within 30 days (lower risk)

In [ ]:
# ── Look at the current distribution of the target variable ───────────────────
print('Current readmitted value counts:')
print(df['readmitted'].value_counts())
print()
print('As percentages:')
print(df['readmitted'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

In [ ]:
# ── Visualize the class distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Chart 1: Original 3-class distribution
df['readmitted'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#3498db','#e74c3c'],
                                      edgecolor='black', rot=0)
axes[0].set_title('Original: 3-Class Readmission', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Readmitted')
axes[0].set_ylabel('Number of Encounters')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{bar.get_height():,.0f}', ha='center', fontsize=10)

# ── Create the binary target variable ────────────────────────────────────────
# If readmitted == '<30' → 1 (positive class = high risk)
# Otherwise              → 0 (negative class = not readmitted within 30 days)
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

# Chart 2: New binary distribution
labels = ['Not Readmitted\nwithin 30 days (0)', 'Readmitted\nwithin 30 days (1)']
counts = df['readmitted_30'].value_counts().sort_index()
colors = ['#3498db', '#e74c3c']
bars = axes[1].bar(labels, counts.values, color=colors, edgecolor='black')
axes[1].set_title('New Binary Target: 30-Day Readmission', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Encounters')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{bar.get_height():,.0f}\n({bar.get_height()/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)

plt.tight_layout()
plt.suptitle('Target Variable Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.savefig('01_target_variable.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n⚠️  Class Imbalance Alert!')
print(f'   Positive class (readmitted <30): {df["readmitted_30"].sum():,} ({df["readmitted_30"].mean()*100:.1f}%)')
print(f'   Negative class (not readmitted): {(df["readmitted_30"]==0).sum():,} ({(1-df["readmitted_30"].mean())*100:.1f}%)')
print(f'\n   → We will need to handle this imbalance before modeling!')

---
## STEP 3 — Handle the `?` Problem (Missing Values)

**What:** This dataset uses `?` as a placeholder for missing values instead of leaving cells blank. We need to convert all `?` to Python's official missing value format: `NaN` (Not a Number).

**Why:** Python doesn't know that `?` means "missing" — it just treats it as a regular text value. If we leave it, our models will crash or give wrong results. Converting to `NaN` lets pandas and sklearn handle them properly.

**What to look for:** Which columns have a lot of missing values? Columns with >50% missing are usually dropped.

In [ ]:
# ── Replace '?' with NaN everywhere ──────────────────────────────────────────
# Before replacement
question_marks_before = (df == '?').sum().sum()
print(f'❓ Question marks found before replacement: {question_marks_before:,}')

df.replace('?', np.nan, inplace=True)  # inplace=True means: change the df itself, not a copy

# After replacement
question_marks_after = (df == '?').sum().sum()
print(f'✅ Question marks after replacement: {question_marks_after}')
print(f'   All ? values are now NaN (proper missing value format)')

In [ ]:
# ── Count and visualize missing values per column ─────────────────────────────
missing = df.isnull().sum()                         # Count NaN in each column
missing_pct = (missing / len(df) * 100).round(1)   # Convert to percentage
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# Only show columns that actually have missing values
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('Columns with missing values:')
print(missing_df.to_string())

# ── Visualize ────────────────────────────────────────────────────────────────
if len(missing_df) > 0:
    colors = ['#e74c3c' if p > 50 else '#f39c12' if p > 20 else '#3498db'
              for p in missing_df['Missing %']]
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(missing_df.index, missing_df['Missing %'], color=colors, edgecolor='black')
    ax.axvline(x=50, color='red', linestyle='--', label='50% threshold (drop zone)', linewidth=2)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    ax.legend()
    for bar, pct in zip(bars, missing_df['Missing %']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{pct}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('02_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()
    print('🔴 Red bars (>50% missing) = DROP these columns')
    print('🟡 Orange bars (20-50% missing) = Decide: drop or impute with Unknown')
    print('🔵 Blue bars (<20% missing) = IMPUTE (fill in the blanks)')

In [ ]:
# ── Decision: What to do with each missing column ────────────────────────────

# 1) DROP 'weight' — ~97% missing. Useless for modeling.
if 'weight' in df.columns:
    df.drop(columns=['weight'], inplace=True)
    print('🗑️  Dropped: weight (97%+ missing)')

# 2) DROP 'payer_code' — ~40% missing + low predictive value
if 'payer_code' in df.columns:
    df.drop(columns=['payer_code'], inplace=True)
    print('🗑️  Dropped: payer_code (40%+ missing, low predictive value)')

# 3) IMPUTE 'race' — ~2% missing. Fill with 'Unknown'
if 'race' in df.columns:
    df['race'] = df['race'].fillna('Unknown')
    print('✏️  Imputed: race → filled missing with "Unknown"')

# 4) IMPUTE 'medical_specialty' — ~49% missing. Group unknowns together.
if 'medical_specialty' in df.columns:
    df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
    print('✏️  Imputed: medical_specialty → filled missing with "Unknown"')

# 5) IMPUTE diagnosis columns — ~1% missing
for col in ['diag_1', 'diag_2', 'diag_3']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')
        print(f'✏️  Imputed: {col} → filled missing with "Unknown"')

print()
# ── Verify: how many missing values remain? ───────────────────────────────────
remaining_missing = df.isnull().sum().sum()
print(f'📊 Total missing values remaining: {remaining_missing}')
if remaining_missing == 0:
    print('✅ No missing values left!')
else:
    print('⚠️  Still have missing values — review the columns above')
    print(df.isnull().sum()[df.isnull().sum() > 0])

---
## STEP 4 — Remove Invalid Records

**What:** Remove patient records that are logically impossible to be readmitted.

**Why:** Some patients were discharged to hospice, died, or were transferred out. If a patient died during their stay, they physically cannot be readmitted 30 days later — including them would corrupt our model.

Also: this dataset has multiple encounters per patient. We'll keep only the **first encounter** per patient to avoid data leakage (where the model accidentally learns from future information).

In [ ]:
# ── Check what discharge disposition values exist ─────────────────────────────
# These are coded numbers. Key ones to remove:
# 11 = Expired (patient died)
# 13 = Hospice / medical facility
# 14 = Hospice / home
# 19 = Expired in medical facility
# 20 = Expired at home
# 21 = Expired / unknown

print('Discharge disposition value counts (before filtering):')
print(df['discharge_disposition_id'].value_counts().sort_index())

rows_before = len(df)

# Remove encounters where patient died or went to hospice
# These patients CANNOT be readmitted
invalid_dispositions = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(invalid_dispositions)]

rows_after_dispo = len(df)
print(f'\n🗑️  Removed {rows_before - rows_after_dispo:,} encounters (death/hospice discharges)')

In [ ]:
# ── Keep only first encounter per patient ─────────────────────────────────────
# 'patient_nbr' is the unique patient ID
# Multiple rows for the same patient = multiple hospital visits
# We keep only the FIRST visit to avoid data leakage

print(f'Encounters before deduplication: {len(df):,}')
print(f'Unique patients: {df["patient_nbr"].nunique():,}')
print(f'Avg encounters per patient: {len(df)/df["patient_nbr"].nunique():.2f}')

# Keep first encounter per patient (sort by encounter_id ascending)
df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')

print(f'\nEncounters after deduplication: {len(df):,}')
print(f'✅ Each row now represents ONE unique patient')

---
## STEP 5 — Explore Key Variables (EDA)

**What:** Exploratory Data Analysis (EDA) — look at distributions of important variables and how they relate to readmission.

**Why:** Before building a model, we need to understand the data. Which patient types are most likely to be readmitted? What patterns exist? EDA guides feature engineering decisions and helps spot problems early.

In [ ]:
# ── Readmission rate by age group ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Patient count by age
age_counts = df['age'].value_counts().sort_index()
age_counts.plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='black', rot=45)
axes[0].set_title('Patient Count by Age Group', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')

# Chart 2: 30-day readmission RATE by age group
readmit_by_age = df.groupby('age')['readmitted_30'].mean().mul(100).round(1)
bars = readmit_by_age.plot(kind='bar', ax=axes[1], color='#e74c3c', edgecolor='black', rot=45)
axes[1].set_title('30-Day Readmission Rate by Age', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Readmission Rate (%)')
axes[1].axhline(y=df['readmitted_30'].mean()*100, color='navy', linestyle='--',
                label=f'Overall avg: {df["readmitted_30"].mean()*100:.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('03_readmission_by_age.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Note how readmission rates vary across age groups.')
print('   Older patients (60-90) tend to have higher readmission risk.')

In [ ]:
# ── Readmission rate by number of prior inpatient visits ──────────────────────
# 'number_inpatient' = number of inpatient visits in the year before this encounter
# This is often one of the STRONGEST predictors of readmission

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clip at 5+ to keep chart readable
df['num_inpatient_capped'] = df['number_inpatient'].clip(upper=5)
labels = {0:'0',1:'1',2:'2',3:'3',4:'4',5:'5+'}

readmit_by_inpatient = df.groupby('num_inpatient_capped')['readmitted_30'].mean().mul(100).round(1)
count_by_inpatient = df['num_inpatient_capped'].value_counts().sort_index()

count_by_inpatient.rename(labels).plot(kind='bar', ax=axes[0], color='#9b59b6',
                                        edgecolor='black', rot=0)
axes[0].set_title('Patient Count by Prior Inpatient Visits', fontsize=12, fontweight='bold')
axes[0].set_xlabel('# Prior Inpatient Visits (Past Year)')
axes[0].set_ylabel('Count')

readmit_by_inpatient.rename(labels).plot(kind='bar', ax=axes[1], color='#e74c3c',
                                          edgecolor='black', rot=0)
axes[1].set_title('30-Day Readmission Rate by Prior Visits', fontsize=12, fontweight='bold')
axes[1].set_xlabel('# Prior Inpatient Visits (Past Year)')
axes[1].set_ylabel('Readmission Rate (%)')
axes[1].axhline(y=df['readmitted_30'].mean()*100, color='navy', linestyle='--',
                label='Overall avg')
axes[1].legend()

plt.tight_layout()
plt.savefig('04_readmission_by_prior_visits.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Key Insight: Patients with MORE prior hospital visits are MUCH more likely to be readmitted.')
print('   This will likely be one of your top predictors!')

In [ ]:
# ── Distribution of key numeric features ──────────────────────────────────────
numeric_features = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                    'num_medications', 'number_diagnoses', 'number_emergency',
                    'number_inpatient', 'number_outpatient']

# Only use columns that actually exist in the dataset
numeric_features = [c for c in numeric_features if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Count')
    axes[i].axvline(df[col].median(), color='red', linestyle='--',
                    label=f'Median: {df[col].median():.1f}')
    axes[i].legend(fontsize=8)

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Key Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('05_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Look for skewed distributions — these may need log transformation for some models.')

---
## STEP 6 — ICD-9 Diagnosis Code Grouping

**What:** The `diag_1`, `diag_2`, `diag_3` columns contain ICD-9 diagnosis codes — there are thousands of unique codes. We'll group them into ~18 broader medical categories.

**Why:** A machine learning model can't use 10,000 unique codes effectively. Grouping them into categories like 'Circulatory', 'Respiratory', 'Diabetes', etc. preserves clinical meaning while making the data manageable. This also directly maps to HRRP-qualifying conditions.

**Why this matters for HRRP:** Heart Failure = Circulatory | COPD = Respiratory | Pneumonia = Respiratory | AMI = Circulatory

In [ ]:
# ── Function to map ICD-9 codes to clinical categories ────────────────────────
def map_icd9_to_category(code):
    """
    Maps an ICD-9 code (as a string) to a clinical category.
    Based on the Clinical Classification Software (CCS) groupings.
    Returns a string label like 'Circulatory', 'Respiratory', etc.
    """
    if pd.isna(code) or code == 'Unknown':
        return 'Unknown'
    
    # Some codes start with 'V' (supplementary) or 'E' (external causes)
    if str(code).startswith('V'):
        return 'Supplementary'
    if str(code).startswith('E'):
        return 'External Causes'
    
    # Try to convert to a number for range-based mapping
    try:
        code_num = float(str(code).split('.')[0])  # Take integer part only
    except:
        return 'Unknown'
    
    # ICD-9 numeric ranges → clinical categories
    if 001 <= code_num <= 139:   return 'Infectious & Parasitic'
    if 140 <= code_num <= 239:   return 'Neoplasms'
    if 240 <= code_num <= 279:   return 'Endocrine/Metabolic'   # Includes Diabetes (250.xx)
    if 280 <= code_num <= 289:   return 'Blood Disorders'
    if 290 <= code_num <= 319:   return 'Mental Disorders'
    if 320 <= code_num <= 389:   return 'Nervous System'
    if 390 <= code_num <= 459:   return 'Circulatory'           # HF (428), AMI (410)
    if 460 <= code_num <= 519:   return 'Respiratory'           # COPD (491-496), Pneumonia (480-486)
    if 520 <= code_num <= 579:   return 'Digestive'
    if 580 <= code_num <= 629:   return 'Genitourinary'
    if 630 <= code_num <= 679:   return 'Pregnancy/Childbirth'
    if 680 <= code_num <= 709:   return 'Skin'
    if 710 <= code_num <= 739:   return 'Musculoskeletal'       # Hip/Knee (820, 821)
    if 740 <= code_num <= 759:   return 'Congenital'
    if 760 <= code_num <= 779:   return 'Perinatal'
    if 780 <= code_num <= 799:   return 'Symptoms/Signs'
    if 800 <= code_num <= 999:   return 'Injury/Poisoning'
    
    return 'Other'

# Apply the function to all three diagnosis columns
for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    if diag_col in df.columns:
        new_col = diag_col + '_category'
        df[new_col] = df[diag_col].apply(map_icd9_to_category)
        print(f'✅ Created {new_col}')

print('\nUnique categories in diag_1_category:')
print(df['diag_1_category'].value_counts())

In [ ]:
# ── Visualize: Readmission rate by primary diagnosis category ─────────────────
if 'diag_1_category' in df.columns:
    readmit_by_diag = df.groupby('diag_1_category')['readmitted_30'].agg(['mean','count'])
    readmit_by_diag.columns = ['Readmission Rate', 'Count']
    readmit_by_diag['Readmission Rate'] = readmit_by_diag['Readmission Rate'].mul(100).round(1)
    readmit_by_diag = readmit_by_diag[readmit_by_diag['Count'] > 100]  # Only categories with >100 patients
    readmit_by_diag = readmit_by_diag.sort_values('Readmission Rate', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(readmit_by_diag.index, readmit_by_diag['Readmission Rate'],
                   color='#e74c3c', edgecolor='black', alpha=0.8)
    ax.axvline(x=df['readmitted_30'].mean()*100, color='navy', linestyle='--',
               label=f'Overall avg: {df["readmitted_30"].mean()*100:.1f}%', linewidth=2)
    ax.set_xlabel('30-Day Readmission Rate (%)')
    ax.set_title('Readmission Rate by Primary Diagnosis Category', fontsize=13, fontweight='bold')
    ax.legend()
    for bar, (_, row) in zip(bars, readmit_by_diag.iterrows()):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{row["Readmission Rate"]}% (n={row["Count"]:,})', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('06_readmission_by_diagnosis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('💡 HRRP-relevant categories: Circulatory (HF, AMI) and Respiratory (COPD, Pneumonia)')

---
## STEP 7 — Encode Categorical Variables

**What:** Convert text/category columns into numbers that machine learning models can understand.

**Why:** Machine learning models work with numbers, not text. `gender = 'Male'` means nothing to an algorithm, but `gender = 1` does. There are different encoding methods for different types of categories:

- **Binary encoding** (0/1): For 2-category columns like gender
- **Ordinal encoding** (0,1,2,...): For ordered categories like age groups
- **One-Hot encoding**: For unordered categories — creates a new column for each category

In [ ]:
# ── 7a: Binary encode GENDER ──────────────────────────────────────────────────
print('Gender value counts before encoding:')
print(df['gender'].value_counts())

# Remove the tiny 'Unknown/Invalid' group (less than 3 patients usually)
df = df[df['gender'].isin(['Male', 'Female'])]

# Encode: Female = 0, Male = 1
df['gender_encoded'] = (df['gender'] == 'Male').astype(int)
print(f'\n✅ Gender encoded: Female=0, Male=1')
print(df['gender_encoded'].value_counts())

In [ ]:
# ── 7b: Ordinal encode AGE ────────────────────────────────────────────────────
# Age is already in ordered decade buckets: [0-10) [10-20) ... [90-100)
# We convert these to numbers 0-9 so the model understands the ORDER

print('Unique age values (should be decade buckets):')
print(sorted(df['age'].unique()))

age_mapping = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3,
    '[40-50)': 4, '[50-60)': 5, '[60-70)': 6, '[70-80)': 7,
    '[80-90)': 8, '[90-100)': 9
}

df['age_encoded'] = df['age'].map(age_mapping)

print('\n✅ Age encoded (0=youngest, 9=oldest):')
print(df[['age', 'age_encoded']].drop_duplicates().sort_values('age_encoded').to_string(index=False))

In [ ]:
# ── 7c: One-Hot encode RACE ───────────────────────────────────────────────────
# One-Hot encoding creates a NEW COLUMN for each category
# Example: race='Caucasian' → race_Caucasian=1, race_AfricanAmerican=0, etc.
# We drop one category to avoid the 'dummy variable trap' (multicollinearity)

print('Race value counts:')
print(df['race'].value_counts())

# pd.get_dummies does all the work — drop_first=True removes one to avoid multicollinearity
race_dummies = pd.get_dummies(df['race'], prefix='race', drop_first=True)
df = pd.concat([df, race_dummies], axis=1)

print(f'\n✅ Race one-hot encoded — created {len(race_dummies.columns)} new columns:')
print(race_dummies.columns.tolist())

In [ ]:
# ── 7d: Group & encode DISCHARGE DISPOSITION ──────────────────────────────────
# This is a key predictor! Where was the patient sent after discharge?
# We'll group the ~30 codes into 5 meaningful groups

def map_discharge(disp_id):
    """
    Maps discharge_disposition_id to a readable group:
    - Home: discharged to home (best outcome, lower risk)
    - Home with Services: home with home health / outpatient care
    - SNF: Skilled Nursing Facility (higher risk for readmission)
    - AMA: Against Medical Advice (very high risk!)
    - Rehab/Long-term: Rehab facility or long-term care
    - Other: Everything else
    """
    if disp_id in [1]:             return 'Home'
    if disp_id in [6, 8]:          return 'Home_with_Services'
    if disp_id in [3, 31]:         return 'SNF'
    if disp_id in [7]:             return 'AMA'
    if disp_id in [4, 5, 24]:      return 'Rehab_LongTerm'
    return 'Other'

df['discharge_group'] = df['discharge_disposition_id'].apply(map_discharge)

# Now one-hot encode the grouped column
discharge_dummies = pd.get_dummies(df['discharge_group'], prefix='discharge', drop_first=True)
df = pd.concat([df, discharge_dummies], axis=1)

print('✅ Discharge disposition grouped and encoded')
print('\nReadmission rate by discharge group:')
print(df.groupby('discharge_group')['readmitted_30'].agg(['mean','count'])
      .rename(columns={'mean':'Readmit Rate','count':'Count'})
      .assign(**{'Readmit Rate': lambda x: (x['Readmit Rate']*100).round(1).astype(str) + '%'})
      .sort_values('Count', ascending=False).to_string())

In [ ]:
# ── 7e: Encode MEDICATION columns ─────────────────────────────────────────────
# There are 24 medication columns (insulin, metformin, etc.)
# Each has values: 'No', 'Steady', 'Up', 'Down'
# We'll simplify: 'No' = 0 (not prescribed), anything else = 1 (prescribed/changed)

med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
            'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
            'miglitol', 'troglitazone', 'tolazamide', 'examide',
            'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone']

# Only encode columns that exist in our dataframe
med_cols_present = [c for c in med_cols if c in df.columns]

for col in med_cols_present:
    df[col + '_encoded'] = (df[col] != 'No').astype(int)

print(f'✅ Encoded {len(med_cols_present)} medication columns')
print('   Values: 0 = not prescribed, 1 = prescribed or dosage changed')
print(f'\nExample — insulin:')
print(f'  Original: {df["insulin"].value_counts().to_dict()}')
if 'insulin_encoded' in df.columns:
    print(f'  Encoded:  {df["insulin_encoded"].value_counts().to_dict()}')

---
## STEP 8 — Select Final Features for Modeling

**What:** Pick which columns to include in the model.

**Why:** We can't (and shouldn't) throw every column into the model. We need to:
1. **Remove ID/administrative columns** — patient ID, encounter ID are not predictive
2. **Remove original columns** we've already encoded (keep the encoded versions)
3. **Select clinically meaningful features** that align with the HRRP problem

Think of feature selection as deciding which "evidence" to hand to your predictive model.

In [ ]:
# ── Define the final feature set ──────────────────────────────────────────────

# NUMERIC features (already in number format — no encoding needed)
numeric_features = [
    'time_in_hospital',      # Length of stay in days
    'num_lab_procedures',    # Number of lab tests done
    'num_procedures',        # Number of procedures performed
    'num_medications',       # Number of medications prescribed
    'number_outpatient',     # Prior outpatient visits
    'number_emergency',      # Prior ER visits
    'number_inpatient',      # Prior inpatient visits (STRONG PREDICTOR)
    'number_diagnoses',      # Number of diagnoses recorded
]

# ENCODED features we created above
encoded_features = [
    'age_encoded',           # Age group (0-9 scale)
    'gender_encoded',        # 0=Female, 1=Male
]

# One-hot encoded race columns (all columns that start with 'race_')
race_features = [c for c in df.columns if c.startswith('race_')]

# One-hot encoded discharge columns
discharge_features = [c for c in df.columns if c.startswith('discharge_')]

# Diagnosis category columns
diag_features = [c for c in df.columns if c.endswith('_category')]

# Medication encoded columns
med_encoded_features = [c for c in df.columns if c.endswith('_encoded')
                        and c not in ['gender_encoded', 'age_encoded']]

# Combine all feature lists
all_features = (numeric_features + encoded_features + race_features +
                discharge_features + med_encoded_features)

# Only keep features that actually exist in our dataframe
all_features = [f for f in all_features if f in df.columns]

print(f'📊 Total features selected for modeling: {len(all_features)}')
print(f'   Numeric: {len(numeric_features)} columns')
print(f'   Encoded (age/gender): {len(encoded_features)} columns')
print(f'   Race one-hot: {len(race_features)} columns')
print(f'   Discharge one-hot: {len(discharge_features)} columns')
print(f'   Medication encoded: {len(med_encoded_features)} columns')

# Create final X (features) and y (target)
X = df[all_features].copy()
y = df['readmitted_30'].copy()

print(f'\n✅ X shape: {X.shape}  (rows × features)')
print(f'✅ y shape: {y.shape}  (target variable)')
print(f'✅ Positive class rate: {y.mean()*100:.1f}% readmitted within 30 days')

In [ ]:
# ── Handle diagnosis categories (need encoding) ───────────────────────────────
# The diag_category columns are still text — encode them too
for col in diag_features:
    if col in df.columns:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        X = pd.concat([X, dummies], axis=1)

# Final check for any remaining NaN values
nan_count = X.isnull().sum().sum()
if nan_count > 0:
    print(f'⚠️  Found {nan_count} NaN values in X — filling with 0')
    X = X.fillna(0)
else:
    print('✅ No NaN values in feature matrix X')

print(f'\nFinal X shape: {X.shape}')

---
## STEP 9 — Train/Test Split

**What:** Split the data into a **training set** (80%) and a **test set** (20%).

**Why:** This is one of the most important concepts in ML. We train the model on the training set, then evaluate it on the test set — which the model has **never seen before**. This simulates how the model would perform in the real world.

Think of it like studying for an exam: you practice on old homework (training set), but you're graded on a new exam (test set).

The `stratify=y` parameter ensures both splits have the same proportion of positive cases (readmissions).

In [ ]:
# ── Perform the train/test split ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X,           # Features
    y,           # Target variable
    test_size=0.20,        # 20% goes to test set
    random_state=42,       # Makes results reproducible (42 is conventional)
    stratify=y             # Keeps the same class ratio in both splits
)

print('📊 Train/Test Split Results:')
print(f'   Training set:  {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'   Test set:      {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'   Features:      {X_train.shape[1]} columns')
print()
print('Class balance check (stratify worked if these match):')
print(f'   Positive class in TRAINING: {y_train.mean()*100:.1f}%')
print(f'   Positive class in TEST:     {y_test.mean()*100:.1f}%')
print('✅ Both splits have similar class proportions — stratify worked!')

---
## STEP 10 — Feature Scaling

**What:** Standardize numeric features so they all have a mean of 0 and standard deviation of 1.

**Why:** Some features (like `num_lab_procedures`) might range from 0–130, while others (like `gender_encoded`) only go 0–1. If we don't scale, the model might incorrectly think `num_lab_procedures` is more important simply because its numbers are bigger. Scaling puts everything on equal footing.

**Important rule:** Fit the scaler on TRAINING data only, then apply it to test data. Never fit on test data — that would be 'cheating' by letting the model peek at future data.

In [ ]:
# ── Apply StandardScaler to numeric columns only ──────────────────────────────
# We only scale the original numeric features (not binary 0/1 encoded ones)
scale_cols = [c for c in numeric_features if c in X_train.columns]

scaler = StandardScaler()

# ⚠️ CRITICAL: fit() only on TRAINING data, then transform() both sets
# fit() learns the mean and std from training data
# transform() applies that scaling (does NOT relearn from test data)

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])   # Note: only .transform(), not .fit_transform()

print('✅ Feature scaling applied!')
print()
print('Before scaling — time_in_hospital:')
print(f'  Mean: {X_train["time_in_hospital"].mean():.2f}, Std: {X_train["time_in_hospital"].std():.2f}')
print('After scaling — time_in_hospital:')
print(f'  Mean: {X_train_scaled["time_in_hospital"].mean():.4f} (≈0), Std: {X_train_scaled["time_in_hospital"].std():.4f} (≈1)')
print()
print('💡 Note: Random Forest and XGBoost do NOT require scaling.')
print('   Use X_train_scaled for Logistic Regression.')
print('   Use X_train (unscaled) for Random Forest and XGBoost.')

---
## STEP 11 — Handle Class Imbalance

**What:** Address the fact that only ~11% of patients are in the positive class (readmitted within 30 days).

**Why:** If 89% of patients were NOT readmitted, a "dumb" model could achieve 89% accuracy by just predicting 'NOT readmitted' for every single patient — and never correctly identify a high-risk patient. We need to compensate for this imbalance.

We'll demonstrate **oversampling** — creating synthetic copies of the minority class to balance the dataset.

In [ ]:
# ── Manual Oversampling (since imbalanced-learn may not be available) ─────────
# We'll oversample by duplicating minority class rows to achieve a ~1:3 ratio
# (not perfect SMOTE, but a valid academic demonstration of the concept)

print('Class distribution BEFORE balancing:')
print(f'  Not readmitted (0): {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)')
print(f'  Readmitted (1):     {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)')

# Combine X_train and y_train for easy manipulation
train_df = X_train.copy()
train_df['target'] = y_train.values

# Separate majority and minority class
majority = train_df[train_df['target'] == 0]
minority = train_df[train_df['target'] == 1]

# Oversample minority class to reach ~33% of majority size
target_minority_size = len(majority) // 3
minority_oversampled = minority.sample(n=target_minority_size, replace=True, random_state=42)

# Recombine
train_balanced = pd.concat([majority, minority_oversampled]).sample(frac=1, random_state=42)
X_train_balanced = train_balanced.drop(columns=['target'])
y_train_balanced = train_balanced['target']

print('\nClass distribution AFTER oversampling:')
print(f'  Not readmitted (0): {(y_train_balanced==0).sum():,} ({(y_train_balanced==0).mean()*100:.1f}%)')
print(f'  Readmitted (1):     {(y_train_balanced==1).sum():,} ({(y_train_balanced==1).mean()*100:.1f}%)')
print(f'\n✅ Training set size: {len(X_train_balanced):,} rows')
print('\n💡 For your final project: Use SMOTE from imbalanced-learn for higher quality synthetic samples.')
print('   Install with: pip install imbalanced-learn')
print('   Then use: from imblearn.over_sampling import SMOTE')

---
## STEP 12 — Save Preprocessed Data & Summary

**What:** Save the cleaned, preprocessed datasets to CSV files for use in modeling.

**Why:** You don't want to rerun all preprocessing every time you experiment with models. Saving now means you can load clean data directly in your modeling notebook.

In [ ]:
# ── Save preprocessed datasets ────────────────────────────────────────────────
X_train_balanced.to_csv('X_train_balanced.csv', index=False)
y_train_balanced.to_csv('y_train_balanced.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

# Also save unbalanced train for comparison
X_train.to_csv('X_train_original.csv', index=False)
y_train.to_csv('y_train_original.csv', index=False)

print('✅ Saved the following files:')
print('   X_train_balanced.csv  — Training features (oversampled)')
print('   y_train_balanced.csv  — Training labels  (oversampled)')
print('   X_test.csv            — Test features (NEVER touch until final evaluation!)')
print('   y_test.csv            — Test labels   (NEVER touch until final evaluation!)')
print('   X_train_original.csv  — Training features (unbalanced, for comparison)')

In [ ]:
# ── FINAL PREPROCESSING SUMMARY ──────────────────────────────────────────────
print('=' * 65)
print('  PREPROCESSING COMPLETE — SUMMARY')
print('=' * 65)
print()
print(f'ORIGINAL DATASET:     {101766:,} rows × 50 columns')
print(f'FINAL DATASET:        {len(X):,} rows × {X.shape[1]} features')
print()
print('STEPS COMPLETED:')
print('  ✅ Step 1:  Data loaded and inspected')
print('  ✅ Step 2:  Target variable recoded to binary (0/1)')
print('  ✅ Step 3:  ? replaced with NaN; missing values treated')
print('  ✅ Step 4:  Invalid records removed (deaths, hospice)')
print('             Duplicated patient encounters deduplicated')
print('  ✅ Step 5:  EDA — key variable distributions explored')
print('  ✅ Step 6:  ICD-9 codes grouped into clinical categories')
print('  ✅ Step 7:  Categorical variables encoded')
print('             (binary, ordinal, one-hot, medication)')
print('  ✅ Step 8:  Final feature set selected')
print('  ✅ Step 9:  80/20 stratified train/test split')
print('  ✅ Step 10: Feature scaling (StandardScaler)')
print('  ✅ Step 11: Class imbalance addressed (oversampling)')
print('  ✅ Step 12: Preprocessed data saved to CSV')
print()
print('READY FOR MODELING!')
print('  Next step: Train Logistic Regression → Random Forest → XGBoost')
print('  Evaluation: ROC-AUC, Precision-Recall AUC, Confusion Matrix')
print('=' * 65)

---
## 🎯 What's Next?

Now that your data is clean and ready, the next notebook will cover **Model Training & Evaluation**:

```
Phase 1: Baseline Model     → Logistic Regression
Phase 2: Primary Model      → Random Forest
Phase 3: Advanced Model     → XGBoost
Phase 4: Model Comparison   → ROC-AUC curves side by side
Phase 5: Interpretability   → SHAP feature importance
Phase 6: Business Output    → Risk score simulation
```

Load your saved data in the modeling notebook with:
```python
import pandas as pd
X_train = pd.read_csv('X_train_balanced.csv')
y_train = pd.read_csv('y_train_balanced.csv').squeeze()
X_test  = pd.read_csv('X_test.csv')
y_test  = pd.read_csv('y_test.csv').squeeze()
```

---
*HealthBridge Medical Center | Advanced Data Analytics Capstone*